# ASG Airlines — Flight Operations Data Pipeline

This notebook walks through cleaning the raw flight/booking/passenger/payment data,
masking PII, and building the KPIs used in the Power BI dashboard.

I'm running everything locally in Python (pandas + SQLite) rather than Azure — the
brief allows this as an alternative, and it lets me show the actual transformation
logic step by step instead of hiding it behind managed services.

## 1. Load the raw data

The source file has four sheets, not just the flight table the brief describes: `flights`, `bookings`, `passengers`, `payments`. I loaded all four since PII and revenue KPIs need the joins across them.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../raw_data/UseCase_-_Airlines.xlsx")
xl = pd.ExcelFile(RAW)
flights_raw = xl.parse("flights")
bookings_raw = xl.parse("bookings")
passengers_raw = xl.parse("passengers")
payments_raw = xl.parse("payments")

for name, df in [("flights", flights_raw), ("bookings", bookings_raw),
                  ("passengers", passengers_raw), ("payments", payments_raw)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

flights: 1020 rows, 7 columns
bookings: 1000 rows, 9 columns
passengers: 1039 rows, 9 columns
payments: 1000 rows, 4 columns


## 2. Data quality check

Before cleaning anything, I checked what was actually wrong rather than assuming. Here's what turned up:

In [2]:
print("Missing airline values:", flights_raw['airline'].isna().sum() + (flights_raw['airline']=='UNKNOWN').sum())
print("Duplicate flight rows (same id, dep, arr):",
      flights_raw.duplicated(subset=['flight_id','departure_time','arrival_time']).sum())
print("Missing booking status:", bookings_raw['status'].isna().sum())
print("Duplicate passenger IDs:", passengers_raw['passenger_id'].duplicated().sum())
print("Missing payment amounts:", payments_raw['amount'].isna().sum())

overnight = flights_raw[flights_raw['arrival_time'].dt.normalize() != flights_raw['departure_time'].dt.normalize()]
print("Overnight (cross-midnight) flights:", len(overnight))

Missing airline values: 72
Duplicate flight rows (same id, dep, arr): 15
Missing booking status: 45
Duplicate passenger IDs: 39
Missing payment amounts: 48
Overnight (cross-midnight) flights: 125


**Takeaway:** the "corruption" here isn't in the flight_id or timestamp format — those are already clean, typed datetimes. The real issues are missing airline/status/amount values, a handful of duplicate rows, and 125 flights that cross midnight and need their duration handled carefully. I'm calling that out because it would've been easy to invent messier-looking fixes than the data actually needed.

## 3. Cleaning: flights

Key decision: overnight flights aren't broken data — arrival timestamps already carry the correct next-day date, so duration = arrival − departure works as-is. I still added an explicit midnight-rollover check so this doesn't silently break if a future data feed only sends time-of-day instead of full timestamps.

In [3]:
def clean_flights(df):
    df = df.copy()
    df['airline'] = df['airline'].replace('UNKNOWN', np.nan)
    df['airline_missing_flag'] = df['airline'].isna()
    df['airline'] = df['airline'].fillna('Not Recorded')

    df = df.drop_duplicates(subset=['flight_id','departure_time','arrival_time'])

    df['departure_time'] = pd.to_datetime(df['departure_time'])
    df['arrival_time'] = pd.to_datetime(df['arrival_time'])

    crosses_midnight = df['arrival_time'] < df['departure_time']
    df.loc[crosses_midnight, 'arrival_time'] += pd.Timedelta(days=1)

    df['duration_minutes'] = (df['arrival_time'] - df['departure_time']).dt.total_seconds() / 60
    df['is_overnight'] = df['departure_time'].dt.normalize() != df['arrival_time'].dt.normalize()
    df['route'] = df['source'].str.upper() + '-' + df['destination'].str.upper()
    df['duration_anomaly'] = (df['duration_minutes'] <= 0) | (df['duration_minutes'] > 360)
    return df

flights = clean_flights(flights_raw)
flights[['flight_id','airline','route','duration_minutes','is_overnight']].head()

,flight_id,airline,route,duration_minutes,is_overnight
0,SJ010,SpiceJet,CCU-MAA,174.0,True
1,AI155,Air India,BOM-CCU,108.0,True
2,UK094,Vistara,BOM-CCU,105.0,True
3,AI245,Air India,BOM-CCU,156.0,True
4,AI192,Air India,MAA-BOM,299.0,True


## 4. Cleaning: bookings, passengers, payments

Same approach elsewhere — flag rather than silently drop or guess, especially for money and booking status, since dropping those rows would quietly bias the KPIs.

In [4]:
def clean_bookings(df):
    df = df.copy()
    df['status'] = df['status'].fillna('UNKNOWN').str.upper()
    df['status_is_invalid'] = df['status'] == 'INVALID'
    return df.drop_duplicates(subset=['booking_id'])

def clean_passengers(df):
    return df.drop_duplicates(subset=['passenger_id']).assign(last_name=lambda d: d['last_name'].fillna(''))

def clean_payments(df):
    df = df.copy()
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    df['amount_missing_flag'] = df['amount'].isna()
    return df

bookings = clean_bookings(bookings_raw)
passengers = clean_passengers(passengers_raw)
payments = clean_payments(payments_raw)
print(len(bookings), len(passengers), len(payments))

1000 1000 1000


## 5. PII masking

The passenger and booking tables carry Aadhaar numbers, passport numbers, phone, and email — real PII, not a hypothetical. I one-way hash the identifiers that don't need to be human-readable (Aadhaar, passport, emergency contact name) and partially mask the ones that are useful to glance at (email, phone), so support staff can still recognize a contact without seeing the full number.

In [5]:
import hashlib

def mask_value(v, salt='asg_airlines_2026'):
    if pd.isna(v):
        return v
    return hashlib.sha256(f"{salt}{v}".encode()).hexdigest()[:16]

def mask_email(e):
    if pd.isna(e) or '@' not in str(e):
        return e
    user, domain = e.split('@', 1)
    return f"{user[0]}***@{domain}"

def mask_phone(p):
    s = str(p)
    return s[:-6] + 'XXXX' + s[-2:] if len(s) > 6 else 'XXXXXX'

passengers['aadhaar_id'] = passengers['aadhaar_id'].apply(mask_value)
passengers['email'] = passengers['email'].apply(mask_email)
passengers['phone'] = passengers['phone'].apply(mask_phone)
bookings['passport_number'] = bookings['passport_number'].apply(mask_value)
bookings['emergency_contact_phone'] = bookings['emergency_contact_phone'].apply(mask_phone)
bookings['emergency_contact_name'] = bookings['emergency_contact_name'].apply(mask_value)

passengers[['passenger_id','email','phone','aadhaar_id']].head()

,passenger_id,email,phone,aadhaar_id
0,P1000,v***@gmail.com,+91-6896XXXX90,9c1cb29d95fd9822
1,P1001,k***@hotmail.com,+91-6702XXXX97,10fce38c0e4d9ce9
2,P1002,m***@outlook.com,+91-6199XXXX92,371798f201337f71
3,P1003,m***@hotmail.com,+91-8719XXXX51,459c0918f8dc1924
4,P1004,s***@outlook.com,+91-7819XXXX13,2cebb2b526e6a75d


## 6. KPIs

The four required KPIs, plus two I added: revenue by route (joining bookings → payments → flights) and cancellation rate, since a flight ops dashboard without any revenue or reliability signal felt incomplete.

In [6]:
avg_duration_by_route = flights.groupby('route')['duration_minutes'].mean().round(1).sort_values(ascending=False)
route_traffic = flights.groupby('route').size().sort_values(ascending=False)
airline_distribution = flights.groupby('airline').size().sort_values(ascending=False)
duration_anomalies = flights[flights['duration_anomaly']]

confirmed = bookings[bookings['status']=='CONFIRMED']
rev = confirmed.merge(payments, on='booking_id', how='left').merge(flights[['flight_id','route']], on='flight_id', how='left')
revenue_by_route = rev.groupby('route')['amount'].sum().round(2).sort_values(ascending=False)
cancellation_rate = round((bookings['status']=='CANCELLED').mean()*100, 2)

print("Avg duration overall:", round(flights['duration_minutes'].mean(),1), "min")
print("Cancellation rate:", cancellation_rate, "%")
print("Duration anomalies flagged:", len(duration_anomalies))
route_traffic.head()

Avg duration overall: 164.6 min
Cancellation rate: 31.4 %
Duration anomalies flagged: 0


route
BOM-CCU    90
CCU-DEL    72
MAA-BLR    65
BLR-BOM    60
HYD-MAA    57
dtype: int64

**Takeaway:** BOM–CCU is the busiest route by a clear margin (90 flights vs the next closest at 72), and roughly 1 in 3 bookings end up cancelled — that cancellation rate alone is worth a dashboard page on its own. No flights got flagged as duration anomalies once overnight handling was fixed, which tells me the earlier duration inconsistencies were purely an overnight-calculation artifact, not genuinely bad readings.

## 7. Export for Power BI

Writing clean CSVs (and a SQLite copy for anyone who wants to query it directly) so the dashboard connects to a stable, already-validated dataset rather than re-cleaning on load.

In [7]:
OUT = Path("../cleaned_data")
OUT.mkdir(exist_ok=True)

flights.to_csv(OUT / "flights_clean.csv", index=False)
bookings.to_csv(OUT / "bookings_clean.csv", index=False)
passengers.to_csv(OUT / "passengers_clean.csv", index=False)
payments.to_csv(OUT / "payments_clean.csv", index=False)
route_traffic.reset_index().to_csv(OUT / "kpi_route_traffic.csv", index=False)
revenue_by_route.reset_index().to_csv(OUT / "kpi_revenue_by_route.csv", index=False)

print("Done — cleaned files written to", OUT.resolve())

Done — cleaned files written to /home/claude/asg_pipeline/cleaned_data
